In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, HTML

# Απενεργοποίηση του auto-scroll στην έξοδο του notebook
display(HTML("<style>.output_scroll { height: unset !important; }</style>"))

def simulate_filters(beta_val, max_freq_mult):
    clear_output(wait=True)
    
    T = 1.0
    
    # --- 1. Zero-Order Hold (ZOH) Calculations ---
    omega = np.linspace(-max_freq_mult * np.pi / T, max_freq_mult * np.pi / T, 3000)
    omega = np.where(omega == 0, 1e-10, omega)
    
    arg_zoh = omega * T / 2.0
    
    # Η μιγαδική συνάρτηση μεταφοράς του ZOH συμπεριλαμβάνει το πρόσημο του sinc
    # έτσι ώστε το np.angle να δώσει τα σωστά άλματα ±\pi (όπως στο σχ. 9.17 του βιβλίου)
    H0_complex = T * (np.sin(arg_zoh) / arg_zoh) * np.exp(-1j * omega * T / 2.0)
    H0_amp = np.abs(H0_complex)
    H0_phase = np.angle(H0_complex)
    
    t_zoh = np.linspace(-0.5 * T, 2.0 * T, 500)
    h0_t = np.where((t_zoh >= 0) & (t_zoh < T), 1.0, 0.0)

    # --- 2. First-Order Hold (FOH) Calculations ---
    sinc_term = np.sin(arg_zoh) / arg_zoh
    complex_term = (1.0 - beta_val) + beta_val * (1.0 + 1j * omega * T) * sinc_term
    H1_complex = T * complex_term * np.exp(-1j * omega * T) * sinc_term
    H1_amp = np.abs(H1_complex)
    H1_phase = np.angle(H1_complex)
    
    t_foh = np.linspace(-0.5 * T, 2.5 * T, 600)
    h1_t = np.zeros_like(t_foh)
    for i, t in enumerate(t_foh):
        if 0 <= t <= T:
            h1_t[i] = 1.0 + beta_val * (t / T)
        elif T < t <= 2 * T:
            h1_t[i] = 1.0 - beta_val * (t / T)
        else:
            h1_t[i] = 0.0

    # --- 3. Figure Setup (2 rows, 3 cols) ---
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    
    x_axis_normalized = omega / (np.pi / T)
    
    # --- Top Row: ZOH ---
    axes[0, 0].plot(x_axis_normalized, H0_amp, 'b-', linewidth=2)
    axes[0, 0].set_title(r'ZOH: Amplitude $|H_0(j\Omega)|$', fontsize=10, fontweight='bold')
    axes[0, 0].set_xlabel(r'Frequency ($\Omega / (\pi/T)$)', fontsize=9)
    axes[0, 0].set_ylabel('Amplitude', fontsize=9)
    axes[0, 0].set_xlim(-max_freq_mult, max_freq_mult)
    axes[0, 0].grid(True, linestyle='--', alpha=0.6)
    
    axes[0, 1].plot(x_axis_normalized, H0_phase, 'r-', linewidth=1.5)
    axes[0, 1].set_title(r'ZOH: Phase $\angle H_0(j\Omega)$', fontsize=10, fontweight='bold')
    axes[0, 1].set_xlabel(r'Frequency ($\Omega / (\pi/T)$)', fontsize=9)
    axes[0, 1].set_ylabel('Phase [rad]', fontsize=9)
    axes[0, 1].set_xlim(-max_freq_mult, max_freq_mult)
    axes[0, 1].set_ylim(-np.pi - 0.3, np.pi + 0.3)
    axes[0, 1].grid(True, linestyle='--', alpha=0.6)
    
    axes[0, 2].plot(t_zoh / T, h0_t, 'g-', linewidth=2)
    axes[0, 2].set_title('ZOH: Impulse $h_0(t)$', fontsize=10, fontweight='bold')
    axes[0, 2].set_xlabel('Time ($t/T$)', fontsize=9)
    axes[0, 2].set_ylabel('$h_0(t)$', fontsize=9)
    axes[0, 2].set_ylim(-0.2, 1.4)
    axes[0, 2].grid(True, linestyle='--', alpha=0.6)

    # --- Bottom Row: FOH ---
    axes[1, 0].plot(x_axis_normalized, H1_amp, 'b-', linewidth=2)
    axes[1, 0].set_title(r'FOH: Amplitude ($|H_1|$, $\beta$=' + f'{beta_val:.2f})', fontsize=10, fontweight='bold')
    axes[1, 0].set_xlabel(r'Frequency ($\Omega / (\pi/T)$)', fontsize=9)
    axes[1, 0].set_ylabel('Amplitude', fontsize=9)
    axes[1, 0].set_xlim(-max_freq_mult, max_freq_mult)
    axes[1, 0].grid(True, linestyle='--', alpha=0.6)
    
    axes[1, 1].plot(x_axis_normalized, H1_phase, 'r-', linewidth=1.5)
    axes[1, 1].set_title(r'FOH: Phase ($\angle H_1$, $\beta$=' + f'{beta_val:.2f})', fontsize=10, fontweight='bold')
    axes[1, 1].set_xlabel(r'Frequency ($\Omega / (\pi/T)$)', fontsize=9)
    axes[1, 1].set_ylabel('Phase [rad]', fontsize=9)
    axes[1, 1].set_xlim(-max_freq_mult, max_freq_mult)
    axes[1, 1].set_ylim(-np.pi - 0.3, np.pi + 0.3)
    axes[1, 1].grid(True, linestyle='--', alpha=0.6)
    
    axes[1, 2].plot(t_foh / T, h1_t, 'g-', linewidth=2)
    axes[1, 2].set_title(r'FOH: Impulse ($h_1$, $\beta$=' + f'{beta_val:.2f})', fontsize=10, fontweight='bold')
    axes[1, 2].set_xlabel('Time ($t/T$)', fontsize=9)
    axes[1, 2].set_ylabel('$h_1(t)$', fontsize=9)
    axes[1, 2].set_ylim(-0.5, 2.0)
    axes[1, 2].grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

display(Markdown(r"""
### User Guide: Filter Characteristics & Frequency Spans
* **Top Row:** ZOH characteristics (matching Fig. 9.17 with proper phase wrapping due to sinc sign changes).
* **Bottom Row:** FOH characteristics parameterized by **$\beta$**.
* **Controls:** Use the $\beta$ slider and the **Freq Span** slider to explore the complete frequency response.
"""))

beta_slider = widgets.FloatSlider(
    value=1.0, min=0.0, max=1.0, step=0.05, 
    description='Parameter β:', 
    style={'description_width': 'initial'}
)

freq_span_slider = widgets.IntSlider(
    value=4, min=2, max=20, step=2, 
    description='Freq Span (±×π/T):', 
    style={'description_width': 'initial'}
)

ui = widgets.VBox([beta_slider, freq_span_slider])
display(ui)

out = widgets.interactive_output(simulate_filters, {'beta_val': beta_slider, 'max_freq_mult': freq_span_slider})
display(out)